# LEAA Training — Stage 4: `moving_slow`
**Target accuracy:** 70%  |  **Timesteps:** 15,000,000

### Setup Instructions
1. **Runtime → Change runtime type → T4 GPU**
2. Add secrets in the left sidebar (🔑 Secrets):
   - `GITHUB_TOKEN` — your GitHub Personal Access Token
   - `GMAIL_ADDRESS` — your Gmail address *(optional, for notifications)*
   - `GMAIL_APP_PASSWORD` — Gmail App Password *(optional)*
     → [Create App Password](https://myaccount.google.com/apppasswords) (requires 2FA enabled)
3. Run all cells in order
4. When session expires, re-open and run all cells — training auto-resumes


In [ ]:
# Cell 1: Check CPU cores & hardware
import os, platform, psutil

cores_physical = psutil.cpu_count(logical=False)
cores_logical = psutil.cpu_count(logical=True)
ram_gb = psutil.virtual_memory().total / (1024**3)
cpu_name = platform.processor() or "Unknown"

# Try to get CPU model name on Linux
try:
    with open("/proc/cpuinfo") as f:
        for line in f:
            if "model name" in line:
                cpu_name = line.split(":")[1].strip()
                break
except:
    pass

print(f"CPU:            {cpu_name}")
print(f"Physical cores: {cores_physical}")
print(f"Logical cores:  {cores_logical}")
print(f"RAM:            {ram_gb:.1f} GB")
print(f"\nRecommended --num-envs: {cores_logical}")

In [ ]:
# Cell 2: Authenticate & clone repo
from google.colab import userdata
import os, subprocess

TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'Sathvik-Chowdary-Veerapaneni/Language-Embeded-Agent-Action'
CLONE_URL = f'https://{TOKEN}@github.com/{REPO}.git'

if not os.path.exists('/content/leaa'):
    subprocess.run(['git', 'clone', CLONE_URL, '/content/leaa'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='/content/leaa', check=True)

subprocess.run(['git', 'config', 'user.email', 'colab@leaa.bot'], cwd='/content/leaa')
subprocess.run(['git', 'config', 'user.name', 'Colab Training Bot'], cwd='/content/leaa')
subprocess.run(['git', 'remote', 'set-url', 'origin', CLONE_URL], cwd='/content/leaa')
print('✓ Repo ready at /content/leaa')

In [ ]:
# Cell 3: Load email credentials (optional)
# Skip this cell if you don't want email notifications.
from google.colab import userdata

try:
    GMAIL_ADDRESS = userdata.get('GMAIL_ADDRESS')
    GMAIL_APP_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
    print(f'✓ Email notifications enabled → {GMAIL_ADDRESS}')
except Exception:
    GMAIL_ADDRESS = None
    GMAIL_APP_PASSWORD = None
    print('⚠ No email credentials found — notifications disabled')
    print('  Add GMAIL_ADDRESS + GMAIL_APP_PASSWORD to Colab Secrets to enable')

In [ ]:
# Cell 4: Install dependencies
%cd /content/leaa
!pip install -q -r requirements.txt
print('✓ Dependencies installed')

In [ ]:
# Cell 5: Run training
# Runs for up to 11h. Checkpoints sync to GitHub every 30 min.
# Runtime watchdog emails a warning at 10h and stops training at 11h
# so the VM has 1h to finish saving before Colab reclaims it.
# If the session expires, re-run all cells — training resumes from last checkpoint.
%cd /content/leaa
import os

cmd = 'python scripts/colab_train.py --stage 4 --timesteps 15000000 --num-envs 4 --max-runtime-hours 11'

# Append email args if credentials are available
if 'GMAIL_ADDRESS' in dir() and GMAIL_ADDRESS:
    cmd += f' --gmail {GMAIL_ADDRESS} --gmail-password {GMAIL_APP_PASSWORD}'

print(f'Running: {cmd}')
os.system(cmd)

In [ ]:
# Cell 6: (Optional) Evaluate this stage after training
%cd /content/leaa
!python rl_training/evaluate.py \\
    --model rl_training/checkpoints/moving_slow_best.zip \\
    --vecnorm rl_training/checkpoints/vecnormalize_moving_slow_best.pkl \\
    --stage moving_slow \\
    --episodes 200